# BW \#156 Winter Olympics
This week we'll look at data about Winter Olympics, allowing us to make all sorts of interesting comparisons. 

## Data and six questions
This week's data comes from a dataset on GitHub from developer Keith Galli, at https://github.com/KeithGalli/Olympics-Dataset. We'll use several of the files in the "clean-data" section of this repository, specifically : 
- `bios.csv`, with information about the athletes' bio
- `noc_regions.csv`, listing the regions in the Olympic games, what we would normally call "countries", except that there isn't a perfect overlap between the notion of a team and a country 
- `results.csv`, with the results from the Olympic games

## Challenges
The learning goals include working with CSV files, cleaning data, joins, pivot tables and using Polars

- Load each of the 3 files into a pandas df, keeping only the rows for the Winter Olympics when reading `results.csv`. Make sure that the `born_date` and `died_date` columns in `bios.csv`are `datetime` values. How many times has the Winter Olympics taken plae ? How many different athletes have competed ? What country has won the greatest number of medals ? (Show the country name, not the NOC code). What country has won the greatest number of gold medals ? (Again show the country name not the NOC code).
- Repeat all the above with Polars. What is the speed difference for each query ? 

In [1]:
import pandas as pd

In [75]:
bios_df = pd.read_csv(r"https://github.com/KeithGalli/Olympics-Dataset/raw/refs/heads/master/clean-data/bios.csv",
                      parse_dates=['born_date', 'died_date'])
noc_regions_df = pd.read_csv(r"https://raw.githubusercontent.com/KeithGalli/Olympics-Dataset/refs/heads/master/clean-data/noc_regions.csv",
                             index_col='NOC')
results_df = pd.read_csv(r"https://github.com/KeithGalli/Olympics-Dataset/raw/refs/heads/master/clean-data/results.csv")

In [30]:
bios_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145500 entries, 0 to 145499
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   athlete_id    145500 non-null  int64         
 1   name          145500 non-null  object        
 2   born_date     143693 non-null  datetime64[ns]
 3   born_city     110908 non-null  object        
 4   born_region   110908 non-null  object        
 5   born_country  110908 non-null  object        
 6   NOC           145499 non-null  object        
 7   height_cm     106651 non-null  float64       
 8   weight_kg     102070 non-null  float64       
 9   died_date     33940 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 11.1+ MB


In [49]:
results_winter = results_df[results_df['type']=='Winter']
results_winter = results_winter.drop(columns="type")
results_winter.shape

(64509, 10)

## First, how many times has it taken place?

In [54]:
results_winter['year'].nunique()

27

## How many different athletes have competed ?

In [57]:
results_winter['athlete_id'].nunique()

23439

## What country has won the greatest number of medals ? 

In [80]:
(
    results_winter
    .set_index('noc')
    .join(noc_regions_df)
    .groupby('region')['medal']
    .count()
    .nlargest(10)
)

region
Russia            833
Germany           806
Canada            800
USA               787
Norway            603
Finland           516
Sweden            516
Austria           372
Switzerland       353
Czech Republic    258
Name: medal, dtype: int64

## What country has won the greatest number of gold medals ? 

In [89]:
(
    results_winter[results_winter['medal']=='Gold']
    .set_index('noc')
    .join(noc_regions_df)
    .groupby('region')['medal']
    .count()
    .nlargest(10)
)

region
Russia         408
Canada         360
Germany        289
Norway         225
USA            219
Sweden         159
Austria        115
Switzerland    113
Finland         95
South Korea     80
Name: medal, dtype: int64